In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.svm import SVR
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score
import optuna
import joblib

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors

In [2]:
kaggle_df = pd.read_csv(r"..\..\..\data\003_kaggle_competition_data\25261_regression_sample_submission.csv")

#Load all 5 SVR ensemble models
svr_homo_rbf = joblib.load(r'tools_used\regression_svr_homo_rbf.joblib')
svr_homo_poly2 = joblib.load(r'tools_used\regression_svr_homo_poly2.joblib')
svr_homo_poly3 = joblib.load(r'tools_used\regression_svr_homo_poly3.joblib')
#lumo
svr_lumo_rbf = joblib.load(r'tools_used\regression_svr_lumo_rbf.joblib')
svr_lumo_poly2 = joblib.load(r'tools_used\regression_svr_lumo_poly2.joblib')
#scalers
scaler_homo = joblib.load(r'tools_used\regression_scaler_homo.joblib')
scaler_lumo = joblib.load(r'tools_used\regression_scaler_lumo.joblib')

# 4. Load feature names
feature_names_homo = joblib.load(r'tools_used\regression_feature_names_homo.joblib')
print(f"Loaded: feature_names_homo ({len(feature_names_homo)} features)")

feature_names_lumo = joblib.load(r'tools_used\regression_feature_names_lumo.joblib')
print(f"Loaded: feature_names_lumo ({len(feature_names_lumo)} features)")

extreme_clipped = joblib.load(r'tools_used\extreme_clipped.joblib')

Loaded: feature_names_homo (250 features)
Loaded: feature_names_lumo (250 features)


In [3]:
descriptor_names = [desc[0] for desc in Descriptors._descList]
calculator = MoleculeDescriptors.MolecularDescriptorCalculator(descriptor_names)
                                                                        #calculate RDKit descriptors for the whole dataset 
def compute_rdkit_descriptors(smiles):
    """Compute RDKit descriptors for a SMILES string"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return [np.nan] * len(descriptor_names)
    try:
        return list(calculator.CalcDescriptors(mol))
    except:
        return [np.nan] * len(descriptor_names)
                                                                        #calculate bond atom features 
def extract_bond_atom_features(smiles):
    """Extract bond and atom counts"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {f: 0 for f in ['n_bonds', 'n_single_bonds', 'n_double_bonds',
                'n_triple_bonds', 'n_aromatic_bonds', 'count_c', 'count_h',
                'count_o', 'count_n', 'count_s', 'count_f', 'count_cl',
                'count_br', 'count_i', 'count_p', 'count_si']}
    
    bonds = mol.GetBonds()
    atoms = mol.GetAtoms()
    atom_symbols = [atom.GetSymbol() for atom in atoms]
    
    return {
        'n_bonds': len(bonds),
        'n_single_bonds': sum(1 for b in bonds if b.GetBondTypeAsDouble() == 1.0),
        'n_double_bonds': sum(1 for b in bonds if b.GetBondTypeAsDouble() == 2.0),
        'n_triple_bonds': sum(1 for b in bonds if b.GetBondTypeAsDouble() == 3.0),
        'n_aromatic_bonds': sum(1 for b in bonds if b.GetBondTypeAsDouble() == 1.5),
        'count_c': atom_symbols.count('C'),
        'count_h': atom_symbols.count('H'),
        'count_o': atom_symbols.count('O'),
        'count_n': atom_symbols.count('N'),
        'count_s': atom_symbols.count('S'),
        'count_f': atom_symbols.count('F'),
        'count_cl': atom_symbols.count('Cl'),
        'count_br': atom_symbols.count('Br'),
        'count_i': atom_symbols.count('I'),
        'count_p': atom_symbols.count('P'),
        'count_si': atom_symbols.count('Si')
    }


In [4]:
kaggle_descriptors = []
for idx, smiles in enumerate(kaggle_df['smiles']):
    if idx % 50 == 0:
        print(f"     Progress: {idx}/{len(kaggle_df)}...")                                  
    kaggle_descriptors.append(compute_rdkit_descriptors(smiles))

X_kaggle_rdkit = pd.DataFrame(kaggle_descriptors, columns=descriptor_names)

kaggle_bonds = [extract_bond_atom_features(s) for s in kaggle_df['smiles']]
kaggle_bonds_df = pd.DataFrame(kaggle_bonds)

X_kaggle = pd.concat([X_kaggle_rdkit, kaggle_bonds_df], axis=1)
X_kaggle = X_kaggle.loc[:, ~X_kaggle.columns.duplicated()]


print(f" Kaggle features: {X_kaggle.shape}")

def add_interactions(X):
                                                    #add extra features, only interactions between existing features
    X['TPSA_x_MolLogP'] = X['TPSA'] * X['MolLogP']
    X['aromatic_x_hetero'] = X['NumAromaticRings'] * X['NumHeteroatoms']
    X['bonds_x_carbon'] = X['n_bonds'] * X['count_c']
    X['aromatic_bonds_x_carbon'] = X['n_aromatic_bonds'] * X['count_c']
    X['hetero_to_carbon'] = X['NumHeteroatoms'] / (X['count_c'] + 1)
    X['aromatic_bond_ratio'] = X['n_aromatic_bonds'] / (X['n_bonds'] + 1)
    X['nitrogen_to_carbon'] = X['count_n'] / (X['count_c'] + 1)
    return X
X_kaggle = add_interactions(X_kaggle)

def add_chemistry_features(X):                                                        #add chemistry, knowledge-based features, likely useful
    X['pi_electrons'] = X['n_aromatic_bonds'] * 2 + X['n_double_bonds'] * 2
    X['conjugation_score'] = (X['NumAromaticRings'] * X['n_aromatic_bonds']) / (X['HeavyAtomCount'] + 1)
    X['nitrogen_density'] = X['count_n'] / (X['HeavyAtomCount'] + 1)
    X['oxygen_density'] = X['count_o'] / (X['HeavyAtomCount'] + 1)
    X['hetero_density'] = X['NumHeteroatoms'] / (X['HeavyAtomCount'] + 1)
    X['charge_range'] = X['MaxPartialCharge'] - X['MinPartialCharge']
    X['nitrogen_aromatic'] = X['count_n'] * X['NumAromaticRings']
    X['aromatic_polarity'] = X['NumAromaticRings'] * X['TPSA']
    X['hetero_conjugation'] = X['NumHeteroatoms'] * X['n_aromatic_bonds']
    X['unsaturation'] = (X['n_double_bonds'] + X['n_triple_bonds'] + X['n_aromatic_bonds']) / (X['n_bonds'] + 1)
    X = X.fillna(0).replace([np.inf, -np.inf], 0)
    return X
X_kaggle = add_chemistry_features(X_kaggle)
X_kaggle = X_kaggle.fillna(0).replace([np.inf, -np.inf], 0)

     Progress: 0/200...
     Progress: 50/200...
     Progress: 100/200...
     Progress: 150/200...
 Kaggle features: (200, 233)


In [5]:
#clipping extremes
for col in extreme_clipped:
    if col in X_kaggle.columns:
        X_kaggle[col] = X_kaggle[col].clip(upper=1e6)

if extreme_clipped:
    print(f"     Clipped {len(extreme_clipped)} features: {extreme_clipped}")

  

     Clipped 1 features: ['Ipc']


In [6]:
for feat in feature_names_homo:
    if feat not in X_kaggle.columns:
        X_kaggle[feat] = 0

for feat in feature_names_lumo:
    if feat not in X_kaggle.columns:
        X_kaggle[feat] = 0

X_kaggle_homo = X_kaggle[feature_names_homo].copy()
X_kaggle_lumo = X_kaggle[feature_names_lumo].copy()

# Scale with TRAINING scalers (CRITICAL!)
X_kaggle_homo_scaled = scaler_homo.transform(X_kaggle_homo)
X_kaggle_lumo_scaled = scaler_lumo.transform(X_kaggle_lumo)


In [7]:
# HOMO: Predict with all 3 models
homo_pred_rbf = svr_homo_rbf.predict(X_kaggle_homo_scaled)
homo_pred_poly2 = svr_homo_poly2.predict(X_kaggle_homo_scaled)
homo_pred_poly3 = svr_homo_poly3.predict(X_kaggle_homo_scaled)

# Ensemble average (weighted)
homo_predictions = 0.6 * homo_pred_rbf + 0.2 * homo_pred_poly2 + 0.2 * homo_pred_poly3

print(f" HOMO ensemble (3 models: RBF 60%, Poly2 20%, Poly3 20%)")
print(f"     Range: [{homo_predictions.min():.4f}, {homo_predictions.max():.4f}]")
print(f"     Mean: {homo_predictions.mean():.4f} ± {homo_predictions.std():.4f}")

# LUMO: Predict with 2 models
lumo_pred_rbf = svr_lumo_rbf.predict(X_kaggle_lumo_scaled)
lumo_pred_poly2 = svr_lumo_poly2.predict(X_kaggle_lumo_scaled)

# Ensemble average
lumo_predictions = 0.7 * lumo_pred_rbf + 0.3 * lumo_pred_poly2

print(f" LUMO ensemble (2 models: RBF 70%, Poly2 30%)")
print(f"     Range: [{lumo_predictions.min():.4f}, {lumo_predictions.max():.4f}]")
print(f"     Mean: {lumo_predictions.mean():.4f} ± {lumo_predictions.std():.4f}")

 HOMO ensemble (3 models: RBF 60%, Poly2 20%, Poly3 20%)
     Range: [-6.0714, -4.4940]
     Mean: -5.2208 ± 0.2641
 LUMO ensemble (2 models: RBF 70%, Poly2 30%)
     Range: [-4.2709, -2.3679]
     Mean: -3.3531 ± 0.4052


In [8]:
#create the submission df

submission = pd.DataFrame({
    'smiles': kaggle_df['smiles'].values,
    'homo': homo_predictions,
    'lumo': lumo_predictions
})

submission.to_csv(r'tools_used\kaggle_submission_svr_clustering.csv', index=False)

print(f"Saved: kaggle_submission_svr_clustering.csv")
print(f"\n   First 5 predictions:")
print(submission.head().to_string(index=False))

Saved: kaggle_submission_svr_clustering.csv

   First 5 predictions:
                                                       smiles      homo      lumo
[SiH2]1cccc1-c1ccc(-c2cnc(-c3nccc4nsnc34)c3nsnc23)c2c[nH]cc12 -5.399997 -4.178056
      c1[nH]cc2c1c(cc1c3ccccc3c3c4ccccc4oc3c21)-c1nccc2nsnc12 -5.304425 -3.443485
   c1cc2c(scc2o1)-c1ccc(o1)-c1ccc(-c2cccc3c[nH]cc23)c2ccccc12 -4.966062 -2.905929
       c1cc2csc(-c3cc4c(s3)c3cnc5ccc6ccccc6c5c3c3ccccc43)c2s1 -5.366454 -3.020309
                             C1cccc1-c1ccc(-c2ccccc2)c2nsnc12 -5.225517 -3.245371
